In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.5828571428571429, 10: 0.6000000000000001, 20: 0.5921428571428573, 30: 0.5957142857142856, 40: 0.5978571428571429, 50: 0.5985714285714285, 60: 0.5964285714285713, 70: 0.5964285714285714, 80: 0.5885714285714285, 90: 0.5807142857142857, 100: 0.5985714285714284, 110: 0.6042857142857144, 120: 0.6064285714285714, 130: 0.6128571428571429, 140: 0.5971428571428571, 150: 0.5964285714285712, 160: 0.5957142857142856, 170: 0.5914285714285713, 180: 0.594285714285714, 190: 0.62, 200: 0.6000000000000001, 210: 0.5842857142857142, 220: 0.6107142857142858, 230: 0.595, 240: 0.6185714285714285, 250: 0.6228571428571431, 260: 0.6085714285714287, 270: 0.6135714285714287, 280: 0.6169117647058825, 290: 0.6109374999999999, 300: 0.61875}
{0: 0.007491836734693877, 10: 0.005464285714285713, 20: 0.005277551020408164, 30: 0.005874489795918368, 40: 0.007298979591836734, 50: 0.008105102040816327, 60: 0.006505102040816327, 70: 0.007933673469387755, 80: 0.0073693877551020415, 90: 0.006074489795918367, 100: 0.006819